# Dense Vector Embeddings (Sentence-Transformers)

Upgrades the TF-IDF baseline to semantic matching: instead of exact word overlap,
resumes and job descriptions are embedded into dense vectors that capture *meaning*,
so "Java Spring Boot engineer" can match a job description about "software engineering"
even with zero shared words — the exact gap the TF-IDF baseline notebook flagged.

Steps:
1. Load the pretrained `all-MiniLM-L6-v2` Sentence-Transformers model
2. Embed all resumes and all job descriptions
3. Score every resume against every job description with cosine similarity
4. Compare top matches against the TF-IDF baseline

Assumes `preprocess.ipynb` has already been run (so `data/processed/resumes_clean.csv`
and `data/processed/job_descriptions.csv` exist). First run downloads the model
(~80MB) and requires internet access.


In [1]:
%run config.ipynb

Config loaded. Data will be stored under: c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data


## 1. Load cleaned data

In [2]:
import pandas as pd

resumes_clean = pd.read_csv(PROCESSED_RESUMES_CSV)
job_descriptions = pd.read_csv(JOB_DESCRIPTIONS_CSV)

print(f"Loaded {len(resumes_clean)} resumes and {len(job_descriptions)} job descriptions")

Loaded 2481 resumes and 24 job descriptions


## 2. Load the embedding model
`all-MiniLM-L6-v2` is a good default: small (~80MB), fast on CPU, and strong general-purpose
semantic performance — a common choice for this kind of similarity task.

In [4]:
%pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------- ----------- 524.3/740.6 kB 1.8 MB/s eta 0:00:01
   ---------------------------------------- 740.6/740.6 kB 1.8 MB/s  0:00:00
   ---------------------------------------- 0.0/842.9 kB ? eta -:--:--
   ------------ --------------------------- 262.1/842.9 kB ? eta -:--:--
   ---------------------------------------- 842.9/842.9 kB 2.2 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 2.6 MB/s eta 0:


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\RanVic\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print(f"Loaded model: {EMBEDDING_MODEL_NAME}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

C:\Users\RanVic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\RanVic\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\RanVic\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingfac

Loaded model: all-MiniLM-L6-v2
Embedding dimension: 384


C:\Users\RanVic\AppData\Local\Temp\ipykernel_65528\4054145139.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")


## 3. Embed resumes and job descriptions

In [6]:
resume_embeddings = model.encode(
    resumes_clean["clean_text"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,  # so cosine similarity == dot product
)

job_embeddings = model.encode(
    job_descriptions["job_description"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True,
)

print("Resume embeddings shape:", resume_embeddings.shape)
print("Job description embeddings shape:", job_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.36s/it]

Resume embeddings shape: (2481, 384)
Job description embeddings shape: (24, 384)


## 4. Score resumes against a job description

In [7]:
import numpy as np

def top_matches_for_category(category: str, top_k: int = 5) -> pd.DataFrame:
    """Return the top_k resumes ranked by embedding cosine similarity to the
    given job description category. Embeddings are normalized, so a simple
    dot product IS the cosine similarity."""
    job_idx = job_descriptions.index[job_descriptions[COL_CATEGORY] == category]
    if len(job_idx) == 0:
        raise ValueError(f"No job description found for category: {category}")
    job_idx = job_idx[0]

    sims = resume_embeddings @ job_embeddings[job_idx]
    results = resumes_clean.copy()
    results["similarity"] = sims
    return results.sort_values("similarity", ascending=False).head(top_k)[
        [COL_ID, COL_CATEGORY, "similarity"]
    ]

example_category = job_descriptions[COL_CATEGORY].iloc[0]
print(f"Top matches for category: {example_category}\n")
top_matches_for_category(example_category)

Top matches for category: ACCOUNTANT



,ID,Category,similarity
1845,20253563,ACCOUNTANT,0.661433
1866,49204385,ACCOUNTANT,0.629326
1822,21794875,ACCOUNTANT,0.618919
1872,29821051,ACCOUNTANT,0.618180
1914,29050809,ACCOUNTANT,0.608460


## 5. Compare against the TF-IDF baseline
Loads the saved TF-IDF artifacts (from `tfidf_baseline.ipynb`) and ranks the same
category with both methods side by side, so you can see where semantic matching
picks up resumes that exact-word matching missed.

In [9]:
# You must define your base tokenize function here as well if it's missing
# def tokenize(text): 
#     ... 

def tokenizer_joiner(text: str) -> str:
    """TfidfVectorizer expects a callable that returns a string or token list;
    we reuse our own tokenize() so stopword handling stays consistent."""
    return tokenize(text)

In [10]:
import joblib

TFIDF_DIR = PROCESSED_DATA_DIR / "tfidf"

try:
    tfidf_vectorizer = joblib.load(TFIDF_DIR / "tfidf_vectorizer.joblib")
    tfidf_resume_vectors = joblib.load(TFIDF_DIR / "resume_vectors.joblib")
    tfidf_job_vectors = joblib.load(TFIDF_DIR / "job_vectors.joblib")

    from sklearn.metrics.pairwise import cosine_similarity

    job_idx = job_descriptions.index[job_descriptions[COL_CATEGORY] == example_category][0]
    tfidf_sims = cosine_similarity(tfidf_job_vectors[job_idx], tfidf_resume_vectors).flatten()

    comparison = resumes_clean[[COL_ID, COL_CATEGORY]].copy()
    comparison["tfidf_similarity"] = tfidf_sims
    comparison["embedding_similarity"] = resume_embeddings @ job_embeddings[job_idx]

    display(comparison.sort_values("embedding_similarity", ascending=False))
except FileNotFoundError:
    print(
        "TF-IDF artifacts not found — run tfidf_baseline.ipynb first "
        "if you want the side-by-side comparison."
    )

,ID,Category,tfidf_similarity,embedding_similarity
1845,20253563,ACCOUNTANT,0.256223,0.661433
1866,49204385,ACCOUNTANT,0.488704,0.629326
1822,21794875,ACCOUNTANT,0.343510,0.618919
1872,29821051,ACCOUNTANT,0.379829,0.618180
1914,29050809,ACCOUNTANT,0.463218,0.608460
...,...,...,...,...
1771,14554542,ENGINEERING,0.010754,0.016013
2433,13405229,AVIATION,0.008059,0.014719
877,11130200,FITNESS,0.013580,-0.003220
355,28086303,TEACHER,0.011898,-0.012716


## 6. Save embeddings for reuse

In [11]:
EMBEDDINGS_DIR = PROCESSED_DATA_DIR / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

np.save(EMBEDDINGS_DIR / "resume_embeddings.npy", resume_embeddings)
np.save(EMBEDDINGS_DIR / "job_embeddings.npy", job_embeddings)
resumes_clean[[COL_ID, COL_CATEGORY]].to_csv(EMBEDDINGS_DIR / "resume_index.csv", index=False)
job_descriptions[[COL_CATEGORY]].to_csv(EMBEDDINGS_DIR / "job_index.csv", index=False)

print(f"Saved embeddings to {EMBEDDINGS_DIR}")

Saved embeddings to c:\Users\RanVic\Downloads\applicant_screening_sprint2\applicant_screening_nb\data_pipeline\data\processed\embeddings


---
## Notes
- Embeddings are **normalized** (`normalize_embeddings=True`), so cosine similarity
  reduces to a dot product — faster to compute at scale.
- `all-MiniLM-L6-v2` is a solid default; if resumes are highly technical/domain-specific,
  a larger model (`all-mpnet-base-v2`) may improve accuracy at the cost of speed.
- The `*_index.csv` files preserve the row order that lines up with the saved `.npy`
  embedding arrays — needed if you reload them later without recomputing.
